In [19]:
import pandas as pd
import mlflow
from dotenv import load_dotenv
from os import getenv
import warnings
from urllib3.exceptions import InsecureRequestWarning
warnings.simplefilter("ignore", InsecureRequestWarning)


MODEL_URI = "mlflow-artifacts:/29/models/m-3728927031ce43878544f8f5f198b0e4/artifacts/"
CSV_PATH = "../../../data/processed/movenet/csv_output/A1.csv"
OUTPUT_PATH = "final_model_predictions_from_mlflow.csv"



In [13]:

RUN_ID = "f6a917ee57c44533a4ca3729785baa29"

client = mlflow.tracking.MlflowClient()
run = client.get_run(RUN_ID)

print("Tracking URI:", mlflow.get_tracking_uri())
print("Artifact URI:", run.info.artifact_uri)

artifacts = client.list_artifacts(RUN_ID)
for a in artifacts:
    print(a.path, a.is_dir, a.file_size)

model_artifacts = client.list_artifacts(RUN_ID, "model")
for a in model_artifacts:
    print(a.path, a.is_dir, a.file_size)

print("Artifact URI:", run.info.artifact_uri)
model_artifacts = client.list_artifacts(RUN_ID, "model")

Tracking URI: https://172.27.62.170:5001/
Artifact URI: mlflow-artifacts:/29/f6a917ee57c44533a4ca3729785baa29/artifacts
a11_problem_b_final_history.csv False 1532
a11_problem_b_final_sequence_eval.csv False 722
a11_problem_b_final_sequence_eval_summary.csv False 227
a11_problem_b_final_test_predictions.csv False 103319
model/MLmodel False 594
model/conda.yaml False 150
model/python_env.yaml False 114
model/python_model.pkl False 403215
model/requirements.txt False 38
Artifact URI: mlflow-artifacts:/29/f6a917ee57c44533a4ca3729785baa29/artifacts


In [17]:
# load environment variables
load_dotenv("../../../ml.env")
print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiments:", [e.name for e in mlflow.search_experiments()[:5]])

Tracking URI: https://172.27.62.170:5001/
Experiments: ['A11_movenet_exercise_frame_classification_B', 'A11_movenet_exercise_frame_classification', 'A10b_pose_conf_smooth_gru', 'A10_2D_posenet', 'A10_posenet_to_kinect2d']


In [20]:
# Load/download model from MLflow
model = mlflow.pyfunc.load_model(MODEL_URI)

# Load raw MoveNet CSV
df = pd.read_csv(CSV_PATH)



In [21]:
# Predict
predictions = model.predict(df)
predictions

,sequence_id,frame_ref,prob,pred,pred_start,pred_stop
0,sequence_0,4,0.000759,0,81,169
1,sequence_0,5,0.000869,0,81,169
2,sequence_0,6,0.000987,0,81,169
3,sequence_0,7,0.000896,0,81,169
4,sequence_0,8,0.001013,0,81,169
...,...,...,...,...,...,...
216,sequence_0,220,0.000218,0,81,169
217,sequence_0,221,0.000216,0,81,169
218,sequence_0,222,0.000185,0,81,169
219,sequence_0,223,0.000164,0,81,169


In [22]:
predictions.to_csv(OUTPUT_PATH, index=False)
print(predictions.head())
print(f"Saved predictions to {OUTPUT_PATH}")

  sequence_id  frame_ref      prob  pred  pred_start  pred_stop
0  sequence_0          4  0.000759     0          81        169
1  sequence_0          5  0.000869     0          81        169
2  sequence_0          6  0.000987     0          81        169
3  sequence_0          7  0.000896     0          81        169
4  sequence_0          8  0.001013     0          81        169
Saved predictions to final_model_predictions_from_mlflow.csv
